# 음원 분리 데모 — GPU 서버 실행 (S15P31A307-166)

실제 곡 몇 개를 서비스 추론 명령 `ssaplash-chart separate`로 분리해 완성곡·드럼·MR을 비교해 들어 보는 노트북이다. 점수는 내지 않는다. 결과물은 팀 내부 청취용이라 저장소·Wiki에 올리지 않는다. **위에서부터 차례로 실행한다.**

**준비물.** JupyterHub `ssaplash/upload/demo_songs/` 폴더를 만들고 곡 파일을 올린다. mp3·wav·flac·opus는 그대로 되고, m4a는 `ffmpeg -i 곡.m4a 곡.flac`으로 바꿔서 올린다. 이 노트북은 `ssaplash/notebooks/`에 올린다. 코드·`.venv`·분리 모델 가중치 `data/models/bs_roformer_sw`는 서버에 이미 있는 것을 쓴다.

**순서.** 1 설정 → 2 코드·가중치 확인 → 3 곡 목록 확인 → 4 분리(곡당 20~40초) → 5 결과 묶기.

**끊기면.** 1번 셀을 다시 실행하고 4번을 다시 실행한다. 이미 분리한 곡은 건너뛴다.

In [ ]:
# 1. 설정. 지정받은 GPU는 6번
import os
import subprocess
import sys
from pathlib import Path

GPU_ID = "6"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTHONUTF8"] = "1"
os.environ["MPLBACKEND"] = "Agg"  # 커널의 Jupyter 전용 그래프 설정을 분리 명령이 물려받으면 모델 올리기에서 멈춤
HOME = Path.home()
WORK = HOME / "ssaplash"
AI = WORK / "S15P31A307" / "AI"
PY = sys.executable
SONGS = WORK / "upload" / "demo_songs"
OUT = WORK / "results" / "separation_demo"
OUT.mkdir(parents=True, exist_ok=True)
!nvidia-smi -i $GPU_ID --query-gpu=index,name,memory.used,memory.total --format=csv
# 확인: GPU 한 줄(H200). memory.used가 커도 남은 메모리가 10GB 넘으면 됨

In [ ]:
# 2. 코드·가중치 확인. 코드는 덮어 풀지 않고 서버에 있는 그대로 씀
%cd $AI
!ls src/ssaplash_ai/chart/cli.py configs/chart/separation.yaml data/models/bs_roformer_sw
!.venv/bin/python -c "import torch, msst; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
# 확인: 파일 두 개와 BS-Rofo-SW-Fixed.ckpt·BS-Rofo-SW-Fixed.yaml이 찍히고, 마지막 줄 'cuda True'

In [ ]:
# 3. 곡 목록 확인
songs = sorted(p for p in SONGS.iterdir() if p.suffix.lower() in {".mp3", ".wav", ".flac", ".opus", ".ogg"})
for p in songs:
    print(f"{p.name}  {p.stat().st_size / 1e6:.1f}MB")
print(len(songs), "곡")
# 확인: 올린 곡 수만큼 줄이 찍힘. 0곡이면 업로드 폴더 이름을 확인

In [ ]:
# 4. 분리. 곡마다 서비스 추론 명령을 한 번씩 부름. 이미 separation.json이 있는 곡은 건너뜀 (여러 번 쳐도 됨)
for p in songs:
    out = OUT / p.stem
    if (out / "separation.json").exists():
        print("건너뜀", p.name)
        continue
    r = subprocess.run(
        [".venv/bin/ssaplash-chart", "separate", str(p), "--out", str(out), "--device", "cuda"],
        capture_output=True, text=True,
    )
    print(p.name, "종료 코드", r.returncode)
    print(r.stdout.strip() or r.stderr.strip()[-500:])
# 확인: 곡마다 '종료 코드 0'과 길이·처리 시간 요약. 2는 입력 오류(못 읽는 형식·10분 초과·무음), 3은 모델 오류

In [ ]:
# 5. 결과 묶기. results 폴더의 separation_demo.zip을 내려받음 (여러 번 쳐도 됨, 매번 새로 만듦)
!cd $WORK/results && $PY -m zipfile -c separation_demo.zip separation_demo
!ls -la $WORK/results/separation_demo.zip
!md5sum $WORK/results/separation_demo.zip
# 확인: 곡당 30~50MB. 곡마다 drums.flac·accompaniment.flac·separation.json 세 파일